# Ahmed Gaitani code ✨

In [ ]:
import pandas as pd
import numpy as np
import re
import string
import nltk
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
# from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error

import pathlib
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [ ]:
# Load the datasets
train_dataset = pd.read_csv('/kaggle/input/rohlik-orders-forecasting-challenge/train.csv')
test_dataset = pd.read_csv('/kaggle/input/rohlik-orders-forecasting-challenge/test.csv')
testids = test_dataset['id']

In [ ]:
# Preprocess the data
ignore_columns = ['id', 'shutdown', 'mini_shutdown', 'blackout', 'mov_change', 'frankfurt_shutdown', 'precipitation', 'snow', 'user_activity_1', 'user_activity_2']
train_dataset = train_dataset.drop(ignore_columns, axis=1, errors="ignore")
test_dataset = test_dataset.drop(ignore_columns, axis=1, errors="ignore")

STRING_ALMOST_MISSING_COLS = ['holiday_name']
train_dataset[STRING_ALMOST_MISSING_COLS] = train_dataset[STRING_ALMOST_MISSING_COLS].astype(str)
test_dataset[STRING_ALMOST_MISSING_COLS] = test_dataset[STRING_ALMOST_MISSING_COLS].astype(str)
train_dataset[STRING_ALMOST_MISSING_COLS] = train_dataset[STRING_ALMOST_MISSING_COLS].fillna('')
test_dataset[STRING_ALMOST_MISSING_COLS] = test_dataset[STRING_ALMOST_MISSING_COLS].fillna('')

DATE_COLUMNS = ['date']
for _col in DATE_COLUMNS:
    train_date_col = pd.to_datetime(train_dataset[_col], errors='coerce')
    train_dataset[_col + "_year"] = train_date_col.dt.year.fillna(-1)
    train_dataset[_col + "_month"] = train_date_col.dt.month.fillna(-1)
    train_dataset[_col + "_day"] = train_date_col.dt.day.fillna(-1)
    train_dataset[_col + "_day_of_week"] = train_date_col.dt.dayofweek.fillna(-1)
    train_dataset.drop(_col, axis=1, inplace=True)

    test_date_col = pd.to_datetime(test_dataset[_col], errors='coerce')
    test_dataset[_col + "_year"] = test_date_col.dt.year.fillna(-1)
    test_dataset[_col + "_month"] = test_date_col.dt.month.fillna(-1)
    test_dataset[_col + "_day"] = test_date_col.dt.day.fillna(-1)
    test_dataset[_col + "_day_of_week"] = test_date_col.dt.dayofweek.fillna(-1)
    test_dataset.drop(_col, axis=1, inplace=True)

TEXT_COLUMNS = ['holiday_name']
def process_text(__dataset):
    for _col in TEXT_COLUMNS:
        process_text = [t.lower() for t in __dataset[_col]]
        table = str.maketrans('', '', string.punctuation)
        process_text = [t.translate(table) for t in process_text]
        process_text = [re.sub(r'\d+', 'num', t) for t in process_text]
        __dataset[_col] = process_text
    return __dataset

train_dataset = process_text(train_dataset)
test_dataset = process_text(test_dataset)

TARGET_COLUMNS = ['orders']
feature_train = train_dataset.drop(TARGET_COLUMNS, axis=1)
target_train = train_dataset[TARGET_COLUMNS].copy()
if set(TARGET_COLUMNS).issubset(test_dataset.columns.tolist()):
    feature_test = test_dataset.drop(TARGET_COLUMNS, axis=1)
    target_test = test_dataset[TARGET_COLUMNS].copy()
else:
    feature_test = test_dataset

CATEGORICAL_COLS = ['warehouse']
onehot_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
train_encoded = pd.DataFrame(onehot_encoder.fit_transform(feature_train[CATEGORICAL_COLS]), columns=onehot_encoder.get_feature_names_out(), index=feature_train.index)
feature_train = pd.concat([feature_train, train_encoded], axis=1)
feature_train.drop(CATEGORICAL_COLS, axis=1, inplace=True)
test_encoded = pd.DataFrame(onehot_encoder.transform(feature_test[CATEGORICAL_COLS]), columns=onehot_encoder.get_feature_names_out(), index=feature_test.index)
feature_test = pd.concat([feature_test, test_encoded], axis=1)
feature_test.drop(CATEGORICAL_COLS, axis=1, inplace=True)

temp_train_data = feature_train[TEXT_COLUMNS]
temp_test_data = feature_test[TEXT_COLUMNS]
feature_train = feature_train.drop(TEXT_COLUMNS, axis=1).astype(pd.SparseDtype('float64', 0))
feature_test = feature_test.drop(TEXT_COLUMNS, axis=1).astype(pd.SparseDtype('float64', 0))
for _col in TEXT_COLUMNS:
    tfidfvectorizer = TfidfVectorizer(max_features=3000)
    vector_train = tfidfvectorizer.fit_transform(temp_train_data[_col])
    feature_names = ['_'.join([_col, name]) for name in tfidfvectorizer.get_feature_names_out()]
    vector_train = pd.DataFrame.sparse.from_spmatrix(vector_train, columns=feature_names, index=temp_train_data.index)
    feature_train = pd.concat([feature_train, vector_train], axis=1)
    vector_test = tfidfvectorizer.transform(temp_test_data[_col])
    vector_test = pd.DataFrame.sparse.from_spmatrix(vector_test, columns=feature_names, index=temp_test_data.index)
    feature_test = pd.concat([feature_test, vector_test], axis=1)


In [ ]:
standard_scaler = StandardScaler(with_mean=False)
feature_train = pd.DataFrame.sparse.from_spmatrix(standard_scaler.fit_transform(feature_train), columns=feature_train.columns, index=feature_train.index)
feature_test = pd.DataFrame.sparse.from_spmatrix(standard_scaler.transform(feature_test), columns=feature_test.columns, index=feature_test.index)

In [ ]:
# Split the training data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(feature_train, target_train, test_size=0.2, random_state=42)

***Train Model By TensorFlow***

In [ ]:
model = keras.Sequential([
        layers.Dense(64, activation=tf.nn.relu, ),
        layers.Dropout(0.5),
        layers.Dense(64, activation=tf.nn.relu, ),
        layers.Dropout(0.5),
        layers.Dense(1)
    ])

optimizer = tf.keras.optimizers.RMSprop(learning_rate=0.001, rho=0.9)

model.compile(loss='mean_squared_error',
                  optimizer=optimizer,
                  metrics=['mean_absolute_error', 'mean_squared_error'])
   



In [ ]:
# Convert sparse data to dense format
X_train_dense = X_train.sparse.to_dense()
X_val_dense = X_val.sparse.to_dense()

# Standardize the target variable
target_scaler = StandardScaler()
y_train_dense = target_scaler.fit_transform(y_train)
y_val_dense = target_scaler.transform(y_val)

# Train the model
history = model.fit(X_train_dense, y_train_dense, epochs=25, validation_data=(X_val_dense, y_val_dense), verbose=1, )


In [ ]:
import matplotlib.pyplot as plt

# Plot history: MAE
plt.plot(history.history['mean_absolute_error'], label='MAE (training data)')
plt.plot(history.history['val_mean_absolute_error'], label='MAE (validation data)')
plt.title('MAE for training and validation data')
plt.ylabel('MAE value')
plt.xlabel('No. epoch')
plt.legend(loc="upper left")
plt.show()


In [ ]:
# Convert test data to dense format
X_test_dense = feature_test.sparse.to_dense()

In [ ]:
# Make predictions
predictions = model.predict(X_test_dense)

# Inverse transform the predictions
predictions = target_scaler.inverse_transform(predictions)


**submission**

In [ ]:
# Prepare the submission DataFrame
submission = pd.DataFrame({
    'id': testids,
    'orders': predictions.flatten()  # Ensure the predictions are a flat array
})

# Save the submission file
submission.to_csv('submission.csv', index=False)
